# QLoRA dengan Stack Hugging Face: Dua Studi Kasus dan Evaluasi yang Jujur

Notebook ini melatih adapter QLoRA pada model kecil (**Qwen2.5-0.5B-Instruct**, 4-bit NF4) untuk dua studi kasus asisten layanan pelanggan fiktif:
**GadaiKita** (jasa gadai) dan **TeknoMart** (toko elektronik online).
Semuanya jalan di GPU T4 gratis, dengan total waktu run di kisaran belasan menit.

**Pertanyaan yang dijawab lewat pengukuran, bukan asumsi**
1. Berapa parameter LoRA yang sebenarnya dilatih, dan bagaimana ia berubah menurut ukuran model, rank, dan target modul?
2. Berapa memori model 4-bit, dan kenapa penghematannya tidak selalu 4x?
3. Apa yang benar-benar dipelajari model dari data kecil: **format** jawaban, **nama brand**, atau **fakta**?
4. Apakah hasilnya berubah saat epoch ditambah?

**Desain evaluasi**
Setiap adapter diuji pada tiga set prompt dengan tujuan berbeda:

| Set | Isi | Yang diuji |
|---|---|---|
| `train_exact` | 10 pertanyaan persis seperti di data latih | hafalan |
| `paraphrase` | 10 pertanyaan yang sama maknanya, dengan kalimat baru | generalisasi ke redaksi baru |
| `heldout` | 4 pertanyaan tentang hal yang tidak ada di data latih | perilaku format saat fakta tidak diketahui |

Metrik dihitung otomatis dan deterministik (decoding greedy): kepatuhan format, ketepatan nama brand, kemunculan fakta kunci, dan pengulangan kata.
Angka hasil dicetak saat notebook dijalankan; teks markdown tidak memuat angka hasil.

> Runtime: **Runtime → Change runtime type → T4 GPU**.

## 1. Instalasi dan lingkungan

In [1]:
%pip install -q -U transformers peft bitsandbytes trl datasets accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 63.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 68.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 49.7 MB/s eta 0:00:00


In [2]:
import os, gc, re, json, time, random
import torch
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, set_seed

assert torch.cuda.is_available(), "GPU tidak terdeteksi. Ubah runtime ke T4 GPU."
props = torch.cuda.get_device_properties(0)
print(f"GPU        : {props.name}")
print(f"VRAM total : {props.total_memory / 1024**3:.1f} GiB")

from google.colab import drive
drive.mount("/content/drive")
OUT_DIR = "/content/drive/MyDrive/AI-Engineer/LLM/qlora-customer-service"
os.makedirs(OUT_DIR, exist_ok=True)

try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN dimuat dari Colab Secrets.")
except Exception:
    print("HF_TOKEN tidak ditemukan. Lanjut tanpa autentikasi (model publik, cukup).")

GPU        : NVIDIA A100-SXM4-40GB
VRAM total : 39.5 GiB
Mounted at /content/drive
HF_TOKEN dimuat dari Colab Secrets.


## 2. Berapa parameter LoRA yang sebenarnya dilatih?

Untuk modul linear `d_in → d_out`, LoRA menambah `r × (d_in + d_out)` parameter. Jadi jumlahnya bergantung pada **rank (r)**, **modul yang dipasangi**, dan **dimensi model**.
Bagian ini murni perhitungan (tanpa GPU), memakai konfigurasi arsitektur tiga model. Persentase dihitung terhadap parameter base, sehingga sedikit berbeda dari `print_trainable_parameters()` yang membaginya dengan base + LoRA.

In [3]:
def lora_param_count(hidden, inter, n_heads, n_kv_heads, n_layers, r, targets="all"):
    head_dim = hidden // n_heads
    q_out, kv_out = n_heads * head_dim, n_kv_heads * head_dim
    mods = {
        "q_proj": (hidden, q_out), "k_proj": (hidden, kv_out),
        "v_proj": (hidden, kv_out), "o_proj": (q_out, hidden),
        "gate_proj": (hidden, inter), "up_proj": (hidden, inter), "down_proj": (inter, hidden),
    }
    if targets == "all":
        names = list(mods)
    elif targets == "qv":
        names = ["q_proj", "v_proj"]
    else:
        names = list(targets)
    return sum(r * (mods[n][0] + mods[n][1]) for n in names) * n_layers

def linear_params(hidden, inter, n_heads, n_kv_heads, n_layers):
    head_dim = hidden // n_heads
    q_out, kv_out = n_heads * head_dim, n_kv_heads * head_dim
    attn = hidden * q_out + 2 * hidden * kv_out + q_out * hidden
    return n_layers * (attn + 3 * hidden * inter)

def total_params(hidden, inter, n_heads, n_kv_heads, n_layers, vocab, tie=False, qkv_bias=False):
    head_dim = hidden // n_heads
    q_out, kv_out = n_heads * head_dim, n_kv_heads * head_dim
    per_layer_extra = 2 * hidden + ((q_out + 2 * kv_out) if qkv_bias else 0)
    lin = linear_params(hidden, inter, n_heads, n_kv_heads, n_layers)
    return lin + n_layers * per_layer_extra + vocab * hidden * (1 if tie else 2) + hidden

ARCHS = {
    "Qwen2.5-0.5B":         dict(hidden=896,  inter=4864,  n_heads=14, n_kv_heads=2,  n_layers=24, vocab=151936, tie=True,  qkv_bias=True),
    "LLaMA-7B (asli)":      dict(hidden=4096, inter=11008, n_heads=32, n_kv_heads=32, n_layers=32, vocab=32000,  tie=False, qkv_bias=False),
    "Llama-3.1-8B":         dict(hidden=4096, inter=14336, n_heads=32, n_kv_heads=8,  n_layers=32, vocab=128256, tie=False, qkv_bias=False),
}
DIMS = ("hidden", "inter", "n_heads", "n_kv_heads", "n_layers")

rows = []
for name, a in ARCHS.items():
    tot = total_params(**a)
    for r in (4, 8, 16, 64):
        for tg in ("qv", "all"):
            n = lora_param_count(**{k: a[k] for k in DIMS}, r=r, targets=tg)
            rows.append({"model": name, "total_params": f"{tot:,}", "r": r,
                         "target": "q,v saja" if tg == "qv" else "semua linear",
                         "lora_params": f"{n:,}", "persen_trainable": f"{100 * n / tot:.3f}%"})
lora_table = pd.DataFrame(rows)
pd.set_option("display.width", 200)
print(lora_table.to_string(index=False))

          model  total_params  r       target lora_params persen_trainable
   Qwen2.5-0.5B   494,032,768  4     q,v saja     270,336           0.055%
   Qwen2.5-0.5B   494,032,768  4 semua linear   2,199,552           0.445%
   Qwen2.5-0.5B   494,032,768  8     q,v saja     540,672           0.109%
   Qwen2.5-0.5B   494,032,768  8 semua linear   4,399,104           0.890%
   Qwen2.5-0.5B   494,032,768 16     q,v saja   1,081,344           0.219%
   Qwen2.5-0.5B   494,032,768 16 semua linear   8,798,208           1.781%
   Qwen2.5-0.5B   494,032,768 64     q,v saja   4,325,376           0.876%
   Qwen2.5-0.5B   494,032,768 64 semua linear  35,192,832           7.124%
LLaMA-7B (asli) 6,738,415,616  4     q,v saja   2,097,152           0.031%
LLaMA-7B (asli) 6,738,415,616  4 semua linear   9,994,240           0.148%
LLaMA-7B (asli) 6,738,415,616  8     q,v saja   4,194,304           0.062%
LLaMA-7B (asli) 6,738,415,616  8 semua linear  19,988,480           0.297%
LLaMA-7B (asli) 6,738,415

## 3. Memori model 4-bit: perkiraan dan pengukuran

Kuantisasi bitsandbytes hanya mengganti **layer linear**. Embedding (dan `lm_head`) tetap FP16, sehingga penghematan memori tidak sebesar 4x pada model kecil yang embedding-nya besar.
Perkiraan di bawah menghitung tiga komponen: bobot linear di 4-bit (plus konstanta kuantisasi), embedding FP16, dan norm/bias FP16.

In [4]:
def estimate_4bit_gb(hidden, inter, n_heads, n_kv_heads, n_layers, vocab, tie=False, qkv_bias=False):
    head_dim = hidden // n_heads
    q_out, kv_out = n_heads * head_dim, n_kv_heads * head_dim
    lin = linear_params(hidden, inter, n_heads, n_kv_heads, n_layers)
    bits_per_weight = 4 + 8 / 64 + 32 / (64 * 256)      # NF4 + konstanta kuantisasi (double quant)
    emb = vocab * hidden * (1 if tie else 2)
    rest = n_layers * (2 * hidden + ((q_out + 2 * kv_out) if qkv_bias else 0)) + hidden
    return (lin * bits_per_weight / 8 + (emb + rest) * 2) / 1e9

mem_rows = []
for name, a in ARCHS.items():
    fp16 = total_params(**a) * 2 / 1e9
    est  = estimate_4bit_gb(**a)
    emb_share = a["vocab"] * a["hidden"] * (1 if a["tie"] else 2) * 2 / 1e9
    mem_rows.append({"model": name, "fp16_gb": round(fp16, 2), "perkiraan_4bit_gb": round(est, 2),
                     "rasio": round(fp16 / est, 2), "embedding_fp16_gb": round(emb_share, 2)})
print(pd.DataFrame(mem_rows).to_string(index=False))

          model  fp16_gb  perkiraan_4bit_gb  rasio  embedding_fp16_gb
   Qwen2.5-0.5B     0.99               0.46   2.16               0.27
LLaMA-7B (asli)    13.48               3.87   3.49               0.52
   Llama-3.1-8B    16.06               5.70   2.82               2.10


## 4. Memuat model 4-bit

Konfigurasi QLoRA: NF4, double quantization, dan compute dtype FP16 (T4 tidak mendukung bfloat16 secara native).
Setelah dimuat, memori terukur dibandingkan dengan perkiraan di bagian 3.

In [5]:
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
SEED = 42
compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)

def load_base_4bit():
    tok = AutoTokenizer.from_pretrained(MODEL_ID)
    mdl = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb_config, device_map="auto")
    return mdl, tok

model, tokenizer = load_base_4bit()
measured = model.get_memory_footprint() / 1e9
a = ARCHS["Qwen2.5-0.5B"]
fp16_gb = total_params(**a) * 2 / 1e9
print(f"Memori terukur (4-bit)     : {measured:.2f} GB")
print(f"Perkiraan (bagian 3)       : {estimate_4bit_gb(**a):.2f} GB")
print(f"FP16 (rumus)               : {fp16_gb:.2f} GB")
print(f"Rasio FP16 / 4-bit terukur : {fp16_gb / measured:.2f}x")
print(f"Embedding di FP16          : {model.get_input_embeddings().weight.numel() * 2 / 1e9:.2f} GB")
del model
gc.collect(); torch.cuda.empty_cache()

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Memori terukur (4-bit)     : 0.45 GB
Perkiraan (bagian 3)       : 0.46 GB
FP16 (rumus)               : 0.99 GB
Rasio FP16 / 4-bit terukur : 2.19x
Embedding di FP16          : 0.27 GB


## 5. Data untuk dua studi kasus

Setiap studi kasus punya 10 fakta (pasangan tanya-jawab). Format jawaban dibuat konsisten:

```
<salam berisi nama brand>  <jawaban inti>  <penutup baku>
```

Setiap fakta digandakan dengan 6 variasi cara bertanya, sehingga data latih berisi 60 contoh per studi kasus. Keduanya fiktif dan dibuat untuk demo.
Kolom `must_contain` berisi kata kunci fakta yang dipakai untuk pengecekan otomatis.

Pertanyaan `heldout` sengaja ditulis **tanpa** menyebut nama brand secara literal. Kalau nama brand muncul di teks pertanyaan, model dasar yang belum dilatih bisa "benar" di metrik `brand` cuma dengan menggemakan kata dari pertanyaan, bukan karena sudah belajar apa pun — itu membuat baseline epoch 0 menyesatkan.

In [6]:
DOMAINS = {
  "gadaikita": dict(
    brand="GadaiKita", brand_re=r"Gadai[A-Z]\w*",
    salam="Halo, terima kasih telah menghubungi GadaiKita.",
    penutup="Ada lagi yang bisa kami bantu?",
    qa=[
      ("Apa syarat mengajukan gadai emas?", "Syaratnya cukup KTP asli dan barang emas yang akan digadaikan; proses dapat dilakukan di seluruh cabang GadaiKita.", ["KTP"]),
      ("Berapa lama proses pencairan dana gadai?", "Proses pencairan dana di GadaiKita rata-rata hanya 15 menit setelah taksiran barang disetujui.", ["15 menit"]),
      ("Apakah barang saya aman selama digadaikan?", "Barang Anda disimpan di ruang khusus berstandar keamanan tinggi dan diasuransikan penuh selama masa gadai.", ["asuransi"]),
      ("Bagaimana cara memperpanjang masa gadai?", "Perpanjangan dapat dilakukan dengan membayar biaya pemeliharaan di cabang atau melalui aplikasi GadaiKita sebelum jatuh tempo.", ["biaya pemeliharaan"]),
      ("Berapa bunga atau biaya gadai per bulan?", "Biaya pemeliharaan GadaiKita mulai dari 1 persen per 15 hari, tergantung golongan pinjaman.", ["1 persen"]),
      ("Apakah bisa menebus barang sebelum jatuh tempo?", "Tentu bisa; Anda dapat menebus barang kapan saja dan biaya hanya dihitung sesuai masa pakai.", ["kapan saja"]),
      ("Barang apa saja yang bisa digadaikan?", "GadaiKita menerima emas, perhiasan, kendaraan bermotor, serta barang elektronik tertentu.", ["kendaraan bermotor"]),
      ("Bagaimana jika saya terlambat membayar?", "Ada masa tenggang; jika melewati batas, barang akan dilelang dan kelebihan hasil lelang dikembalikan kepada Anda.", ["dilelang"]),
      ("Apakah ada aplikasi untuk cek status gadai?", "Ada; unduh aplikasi GadaiKita untuk memantau status gadai, jatuh tempo, dan pembayaran secara online.", ["unduh"]),
      ("Bisakah gadai diwakilkan orang lain?", "Pengajuan gadai harus dilakukan pemilik barang, namun pembayaran cicilan dapat diwakilkan.", ["pemilik barang"]),
    ],
    paraphrase=[
      "Dokumen apa yang harus saya bawa untuk menggadaikan emas?",
      "Kira-kira dana gadai cair dalam berapa menit?",
      "Kalau barang saya digadaikan, apakah ada jaminan keamanannya?",
      "Kalau mau perpanjang gadai, caranya bagaimana?",
      "Biaya gadainya berapa persen?",
      "Boleh nggak saya tebus barangnya lebih cepat dari jadwal?",
      "Jenis barang apa yang diterima buat digadai?",
      "Apa yang terjadi kalau saya telat bayar?",
      "Ada app buat memantau gadai saya?",
      "Bisa nggak orang lain yang mengurus gadai saya?",
    ],
    heldout=[
      "Apakah ada layanan yang buka di hari Minggu?",
      "Berapa nilai taksiran emas per gram hari ini?",
      "Apakah ada cabang di Surabaya?",
      "Bagaimana cara mengajukan keluhan layanan?",
    ],
  ),
  "teknomart": dict(
    brand="TeknoMart", brand_re=r"Tekno[A-Z]\w*",
    salam="Halo Kak, terima kasih sudah menghubungi TeknoMart Care.",
    penutup="Jika perlu bantuan lebih lanjut, tim CS kami siap membantu melalui live chat aplikasi.",
    qa=[
      ("Berapa lama estimasi pengiriman pesanan saya?", "Estimasi pengiriman TeknoMart adalah 2-4 hari kerja untuk area Jawa dan 4-7 hari kerja untuk luar Jawa.", ["2-4 hari kerja"]),
      ("Bagaimana cara mengajukan retur barang yang rusak?", "Kakak bisa ajukan retur lewat menu Pesanan Saya dalam 7 hari sejak barang diterima, lalu unggah foto kondisi barang.", ["7 hari"]),
      ("Metode pembayaran apa saja yang tersedia?", "TeknoMart menerima transfer bank, kartu kredit/debit, e-wallet, dan cicilan tanpa kartu kredit.", ["e-wallet"]),
      ("Apakah bisa membatalkan pesanan setelah dibayar?", "Pembatalan hanya bisa dilakukan sebelum status pesanan berubah menjadi Dikemas, melalui menu Pesanan Saya.", ["Dikemas"]),
      ("Bagaimana cara melacak status pengiriman?", "Kakak bisa memantau status pengiriman secara real-time di menu Lacak Pesanan menggunakan nomor resi.", ["resi"]),
      ("Apakah produk elektronik di TeknoMart bergaransi resmi?", "Seluruh produk elektronik di TeknoMart bergaransi resmi distributor, dengan kartu garansi disertakan dalam paket.", ["garansi resmi"]),
      ("Bagaimana jika barang yang diterima tidak sesuai pesanan?", "Kakak dapat mengajukan komplain via menu Pusat Bantuan dan kami akan mengirimkan penggantian tanpa biaya tambahan.", ["Pusat Bantuan"]),
      ("Apakah ada biaya pengiriman minimum belanja?", "Gratis ongkir berlaku untuk pembelanjaan minimal Rp150.000 ke seluruh wilayah yang terjangkau kurir rekanan.", ["Rp150.000"]),
      ("Bagaimana cara mengubah alamat pengiriman setelah checkout?", "Perubahan alamat hanya bisa dilakukan sebelum pesanan diproses, dengan menghubungi CS melalui live chat secepatnya.", ["sebelum pesanan diproses"]),
      ("Apakah bisa refund jika pesanan tidak kunjung sampai?", "Refund penuh akan diproses otomatis jika pesanan tidak sampai melebihi batas waktu SLA pengiriman yang tertera.", ["SLA"]),
    ],
    paraphrase=[
      "Kira-kira pesanan saya sampai dalam berapa hari?",
      "Barang yang saya terima rusak, gimana cara retur?",
      "Bayarnya bisa pakai apa saja?",
      "Pesanan yang sudah saya bayar masih bisa dibatalkan?",
      "Di mana saya bisa cek posisi paket saya?",
      "Barang elektronik di sini ada garansi resminya?",
      "Barang yang datang beda dari yang saya pesan, harus bagaimana?",
      "Belanja berapa supaya dapat gratis ongkir?",
      "Saya salah isi alamat setelah checkout, bisa diganti?",
      "Kalau paket tidak sampai-sampai, uang saya dikembalikan?",
    ],
    heldout=[
      "Apakah ada toko offline yang bisa dikunjungi?",
      "Bagaimana cara menggunakan kode voucher?",
      "Apakah bisa bayar di tempat (COD)?",
      "Berapa lama proses refund ke rekening?",
    ],
  ),
}

VARIASI = ["{q}", "{q} Mohon infonya.", "Mau tanya, {q_lower}",
           "Permisi, {q_lower}", "{q} Terima kasih.", "Halo kak, {q_lower}"]

def build_train_rows(dom, seed=SEED):
    rows = []
    for q, ans, _ in dom["qa"]:
        for v in VARIASI:
            question = v.format(q=q, q_lower=q[0].lower() + q[1:])
            reply = f"{dom['salam']} {ans} {dom['penutup']}"
            rows.append({"messages": [{"role": "user", "content": question},
                                      {"role": "assistant", "content": reply}]})
    random.Random(seed).shuffle(rows)
    return rows

for k, d in DOMAINS.items():
    print(k, "-> contoh latih:", len(build_train_rows(d)),
          "| train_exact:", len(d["qa"]), "| paraphrase:", len(d["paraphrase"]), "| heldout:", len(d["heldout"]))

gadaikita -> contoh latih: 60 | train_exact: 10 | paraphrase: 10 | heldout: 4
teknomart -> contoh latih: 60 | train_exact: 10 | paraphrase: 10 | heldout: 4


### Pemeriksaan kebocoran data (wajib lolos sebelum lanjut)

`paraphrase` dan `heldout` dirancang untuk **tidak** muncul kata demi kata di data latih; hanya `train_exact` yang memang sengaja identik (itu yang dipakai untuk mengukur hafalan).
Sel di bawah memverifikasi ini secara terprogram dan langsung berhenti (`AssertionError`) kalau ada tumpang tindih, alih-alih diam-diam menghasilkan evaluasi yang bias.

In [7]:
for dkey, dom in DOMAINS.items():
    train_qs = {r["messages"][0]["content"] for r in build_train_rows(dom)}
    leak_para = [q for q in dom["paraphrase"] if q in train_qs]
    leak_held = [q for q in dom["heldout"] if q in train_qs]
    assert not leak_para, f"{dkey}: prompt paraphrase bocor ke data latih: {leak_para}"
    assert not leak_held, f"{dkey}: prompt heldout bocor ke data latih: {leak_held}"
    print(f"{dkey}: tidak ada kebocoran pada paraphrase ({len(dom['paraphrase'])} prompt) maupun heldout ({len(dom['heldout'])} prompt).")

gadaikita: tidak ada kebocoran pada paraphrase (10 prompt) maupun heldout (4 prompt).
teknomart: tidak ada kebocoran pada paraphrase (10 prompt) maupun heldout (4 prompt).


## 6. Metrik evaluasi

Semua metrik dihitung otomatis dari teks jawaban:

- **format**: jawaban diawali persis dengan salam dan diakhiri persis dengan penutup baku.
- **brand**: semua kata bergaya nama brand di jawaban (misalnya `GadaiKita`, `GadaiEmas`) harus tepat satu, yaitu nama brand yang benar.
- **fact**: semua kata kunci fakta muncul di jawaban (hanya untuk `train_exact` dan `paraphrase`).
- **degenerate**: ada tiga kata berurutan yang muncul 3 kali atau lebih (tanda pengulangan).

Keterbatasan: pengecekan kata kunci tidak menilai apakah kalimatnya benar secara keseluruhan. Jawaban bisa memuat kata kunci tapi salah konteks, atau benar tapi berbeda redaksi. Karena itu contoh jawaban tetap dibaca manual di bagian 8.

In [8]:
def has_repetition(text, n=3, min_count=3):
    words = re.findall(r"\w+", text.lower())
    grams = {}
    for i in range(len(words) - n + 1):
        g = tuple(words[i:i + n])
        grams[g] = grams.get(g, 0) + 1
    return any(c >= min_count for c in grams.values())

def score_answer(ans, dom, must=None):
    a = ans.strip()
    brands = set(re.findall(dom["brand_re"], a))
    out = {
        "format": a.startswith(dom["salam"]) and a.endswith(dom["penutup"]),
        "brand": brands == {dom["brand"]},
        "degenerate": has_repetition(a),
        "fact": None,
    }
    if must:
        out["fact"] = all(k.lower() in a.lower() for k in must)
    return out

def summarize(scores):
    n = len(scores)
    facts = [s["fact"] for s in scores if s["fact"] is not None]
    return {
        "n": n,
        "format": round(100 * sum(s["format"] for s in scores) / n, 1),
        "brand": round(100 * sum(s["brand"] for s in scores) / n, 1),
        "fact": round(100 * sum(facts) / len(facts), 1) if facts else None,
        "degenerate": sum(s["degenerate"] for s in scores),
    }

In [9]:
def generate_batch(model, tok, questions, max_new_tokens=110, batch_size=12):
    tok.padding_side = "left"
    texts = [tok.apply_chat_template([{"role": "user", "content": q}], tokenize=False, add_generation_prompt=True)
             for q in questions]
    outs = []
    for s in range(0, len(texts), batch_size):
        enc = tok(texts[s:s + batch_size], return_tensors="pt", padding=True).to(model.device)
        with torch.no_grad():
            gen = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                                 temperature=None, top_p=None, pad_token_id=tok.eos_token_id)
        outs += tok.batch_decode(gen[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    return [o.strip() for o in outs]

def evaluate_model(model, tok, dom):
    sets = {
        "train_exact": [(q, must) for q, _, must in dom["qa"]],
        "paraphrase":  [(q, must) for q, must in zip(dom["paraphrase"], [m for _, _, m in dom["qa"]])],
        "heldout":     [(q, None) for q in dom["heldout"]],
    }
    answers, summary = {}, {}
    for name, items in sets.items():
        ans = generate_batch(model, tok, [q for q, _ in items])
        answers[name] = ans
        summary[name] = summarize([score_answer(a, dom, m) for a, (_, m) in zip(ans, items)])
    return answers, summary

## 7. Eksperimen: baseline, epoch rendah, epoch tinggi

Untuk setiap studi kasus dijalankan tiga kondisi:
- `0` epoch: model dasar tanpa fine-tuning (baseline).
- `3` epoch: konfigurasi singkat.
- `15` epoch: training lebih lama pada data yang sama.

Seed, data, hyperparameter, dan set evaluasi dibuat identik antar kondisi, jadi perbedaan hasil berasal dari lamanya training.
Tiap kondisi memuat ulang model dari awal. Pada konfigurasi ini loss dihitung pada seluruh token percakapan (`assistant_only_loss` tidak diaktifkan), bukan hanya pada jawaban asisten.

In [10]:
from datasets import Dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

LORA_CFG = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

def make_trainer(model, tok, dataset, epochs):
    cfg = SFTConfig(
        output_dir="/content/qlora-tmp",
        num_train_epochs=epochs,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        lr_scheduler_type="cosine",
        logging_steps=2,
        optim="paged_adamw_8bit",
        max_length=512,
        save_strategy="no",
        report_to="none",
        seed=SEED,
    )
    return SFTTrainer(model=model, args=cfg, train_dataset=dataset, processing_class=tok)

def run_experiment(dkey, epochs):
    dom = DOMAINS[dkey]
    set_seed(SEED)
    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    model, tok = load_base_4bit()
    info = dict(domain=dkey, epochs=epochs, train_loss=None, train_seconds=0.0,
                trainable_params=None, lora_formula_match=None)
    if epochs > 0:
        model = prepare_model_for_kbit_training(model)
        model = get_peft_model(model, LORA_CFG)
        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        expected = lora_param_count(**{k: ARCHS["Qwen2.5-0.5B"][k] for k in DIMS}, r=16, targets="all")
        info.update(trainable_params=trainable, lora_formula_match=(trainable == expected))
        trainer = make_trainer(model, tok, Dataset.from_list(build_train_rows(dom)), epochs)
        t0 = time.time()
        res = trainer.train()
        info.update(train_seconds=round(time.time() - t0, 1), train_loss=round(res.training_loss, 4))
        trainer.save_model(f"{OUT_DIR}/adapter-{dkey}-ep{epochs}")
    model.eval()
    answers, summary = evaluate_model(model, tok, dom)
    info["peak_vram_gb"] = round(torch.cuda.max_memory_allocated() / 1e9, 2)
    del model
    gc.collect(); torch.cuda.empty_cache()
    return info, answers, summary

EPOCH_GRID = [0, 3, 15]
all_info, all_answers, all_summary = [], {}, {}
for dkey in DOMAINS:
    for ep in EPOCH_GRID:
        print(f">>> {dkey} | epoch={ep}")
        info, answers, summary = run_experiment(dkey, ep)
        all_info.append(info); all_answers[(dkey, ep)] = answers; all_summary[(dkey, ep)] = summary
        print("    selesai:", {k: info[k] for k in ("train_loss", "train_seconds", "peak_vram_gb")})

>>> gadaikita | epoch=0


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

    selesai: {'train_loss': None, 'train_seconds': 0.0, 'peak_vram_gb': 0.51}
>>> gadaikita | epoch=3


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Tokenizing train dataset:   0%|          | 0/60 [00:00<?, ? examples/s]

...

Truncating train dataset:   0%|          | 0/60 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/60 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Step,Training Loss
2,3.232425
4,2.005351
6,1.332887
8,0.872661
10,0.600677
12,0.422225
14,0.284477
16,0.167815
18,0.099755
20,0.084276


    selesai: {'train_loss': 0.3314, 'train_seconds': 114.0, 'peak_vram_gb': 3.39}


## 8. Hasil

Tabel di bawah menampilkan, untuk tiap studi kasus, kondisi, dan set evaluasi: persentase jawaban yang patuh format, memakai nama brand yang benar, memuat fakta kunci, serta jumlah jawaban yang mengalami pengulangan.
Setelah tabel, sejumlah jawaban ditampilkan apa adanya agar bisa dinilai langsung.

In [11]:
rows = []
for (dkey, ep), summ in all_summary.items():
    for set_name, m in summ.items():
        rows.append(dict(domain=dkey, epochs=ep, eval_set=set_name, n=m["n"],
                         format_pct=m["format"], brand_pct=m["brand"], fact_pct=m["fact"],
                         degenerate=m["degenerate"]))
results_df = pd.DataFrame(rows)
print(results_df.to_string(index=False))

info_df = pd.DataFrame(all_info)
print("\nRingkasan training:")
print(info_df.to_string(index=False))

results_df.to_csv(f"{OUT_DIR}/eval_results.csv", index=False)
info_df.to_csv(f"{OUT_DIR}/train_info.csv", index=False)

   domain  epochs    eval_set  n  format_pct  brand_pct  fact_pct  degenerate
gadaikita       0 train_exact 10         0.0        0.0       0.0           0
gadaikita       0  paraphrase 10         0.0        0.0       0.0           0
gadaikita       0     heldout  4         0.0        0.0       NaN           0
gadaikita       3 train_exact 10        70.0       90.0       0.0           0
gadaikita       3  paraphrase 10       100.0      100.0      10.0           0
gadaikita       3     heldout  4       100.0      100.0       NaN           0
gadaikita      15 train_exact 10       100.0      100.0     100.0           0
gadaikita      15  paraphrase 10       100.0      100.0      40.0           0
gadaikita      15     heldout  4       100.0      100.0       NaN           0
teknomart       0 train_exact 10         0.0       10.0      10.0           2
teknomart       0  paraphrase 10         0.0        0.0      10.0           0
teknomart       0     heldout  4         0.0        0.0       Na

In [12]:
def show_samples(dkey, ep, set_name, idxs):
    dom = DOMAINS[dkey]
    qs = {"train_exact": [q for q, _, _ in dom["qa"]], "paraphrase": dom["paraphrase"], "heldout": dom["heldout"]}[set_name]
    for i in idxs:
        print(f"[{dkey} | epoch={ep} | {set_name}] {qs[i]}")
        print(f"   -> {all_answers[(dkey, ep)][set_name][i]}\n")

for dkey in DOMAINS:
    for ep in EPOCH_GRID:
        show_samples(dkey, ep, "paraphrase", [0, 2])
        show_samples(dkey, ep, "heldout", [0])

with open(f"{OUT_DIR}/all_answers.json", "w", encoding="utf-8") as f:
    json.dump({f"{d}|ep{e}": a for (d, e), a in all_answers.items()}, f, ensure_ascii=False, indent=2)

[gadaikita | epoch=0 | paraphrase] Dokumen apa yang harus saya bawa untuk menggadaikan emas?
   -> Tentu saja, berikut adalah beberapa dokumen yang dapat Anda gunakan untuk menggadaikan emas:

1. Emas: Anda boleh memakai emas dalam kebanyakan situasi kerja atau acara-acara pribadi.

2. Emas di kamar tidur: Anda boleh memakai emas di kamar tidur jika Anda merasa nyaman dan tidak ada yang terlihat dari luar.

3. Emas di ruang tamu: Anda boleh memak

[gadaikita | epoch=0 | paraphrase] Kalau barang saya digadaikan, apakah ada jaminan keamanannya?
   -> Tentu saja, setiap negara memiliki hak untuk memilih dan mengelola jaminan keamanan mereka sendiri. Jadi, jika Anda berminat dalam hal ini, berikut adalah beberapa langkah yang dapat Anda lakukan:

1. **Pendekatan Kesehatan**: Jika Anda merasa bahwa keamanan rumah atau tempat lainnya tidak cukup, Anda mungkin akan mencoba untuk mengeksploitasi jaringan internet atau teknologi lainnya.

2. **

[gadaikita | epoch=0 | heldout] Apakah ada layana

### Cara membaca hasil

- Bandingkan `train_exact` dengan `paraphrase`. Selisih di antara keduanya menunjukkan seberapa jauh hasil hanya hafalan.
- Bandingkan `format_pct` dan `fact_pct` pada epoch 3 dan epoch 15. Kalau keduanya bergerak berbeda, itu petunjuk bahwa format dan fakta dipelajari dengan kecepatan berbeda.
- Pada `heldout`, fakta tidak ada di data latih, jadi hanya `format_pct` dan `brand_pct` yang bermakna. Baca jawabannya: format yang benar tidak menjamin isi yang benar.
- **`brand_pct` di dua domain sekaligus.** Kalau nama brand (`GadaiKita`, `TeknoMart`) meleset di jawaban meskipun salam dan penutup persis sama seperti data latih, itu tanda entitas literal tidak terkunci oleh SFT data kecil. Kalau pola ini muncul di kedua domain, bukan cuma satu, itu argumen yang lebih kuat untuk kesimpulan "ini sistemik" dibanding kalau cuma satu domain yang bermasalah.
- Ini eksperimen kecil (satu seed, 10 fakta per kasus). Selisih beberapa persen tidak boleh ditafsirkan sebagai perbedaan yang stabil.

## 9. Menyimpan, memuat ulang, dan memasang-lepas adapter

Keunggulan operasional LoRA: yang disimpan hanya adapter. Dua adapter di bawah dimuat ke **satu** base model 4-bit yang sama, lalu dipilih lewat `set_adapter`.
Sel kedua memeriksa apakah keluaran greedy setelah reload sama dengan keluaran saat evaluasi.

In [13]:
from peft import PeftModel

FINAL_EP = max(EPOCH_GRID)
paths = {d: f"{OUT_DIR}/adapter-{d}-ep{FINAL_EP}" for d in DOMAINS}
for d, p in paths.items():
    files = os.listdir(p)
    adapter_file = next(f for f in files if f.startswith("adapter_model"))
    print(f"{d}: {adapter_file} = {os.path.getsize(os.path.join(p, adapter_file)) / 1e6:.1f} MB")

base, tok = load_base_4bit()
swap = PeftModel.from_pretrained(base, paths["gadaikita"], adapter_name="gadaikita")
swap.load_adapter(paths["teknomart"], adapter_name="teknomart")
swap.eval()

question = "Bisa nggak orang lain yang mengurus pesanan saya?"
for name in ("gadaikita", "teknomart"):
    swap.set_adapter(name)
    print(f"[adapter={name}] {generate_batch(swap, tok, [question])[0]}\n")

gadaikita: adapter_model.safetensors = 17.6 MB
teknomart: adapter_model.safetensors = 17.6 MB


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[adapter=gadaikita] Halo, terima kasih telah menghubungi GadaiKita. Pengajuan gadai harus dilakukan pemilik barang, namun pembayaran cicilan dapat diwakilkan. Ada lagi yang bisa kami bantu?

[adapter=teknomart] Halo Kak, terima kasih sudah menghubungi TeknoMart Care. Kakak dapat memantau status pesanan secara real-time di menu Lacak Pesanan untuk menyesuaikan dengan pengiriman yang terbaik. Jika perlu bantuan lebih lanjut, tim CS kami siap membantu melalui live chat aplikasi.



In [14]:
match_rows = []
for dkey in DOMAINS:
    swap.set_adapter(dkey)
    for set_name, qs in (("paraphrase", DOMAINS[dkey]["paraphrase"]), ("heldout", DOMAINS[dkey]["heldout"])):
        again = generate_batch(swap, tok, qs)
        same = sum(a == b for a, b in zip(again, all_answers[(dkey, FINAL_EP)][set_name]))
        match_rows.append(dict(domain=dkey, eval_set=set_name, identik=f"{same}/{len(qs)}"))
print(pd.DataFrame(match_rows).to_string(index=False))
print("\nCatatan: keluaran bisa berbeda tipis karena model saat training melewati prepare_model_for_kbit_training "
      "(beberapa parameter di-upcast), sedangkan model hasil reload tidak.")

   domain   eval_set identik
gadaikita paraphrase    7/10
gadaikita    heldout     3/4
teknomart paraphrase    7/10
teknomart    heldout     2/4

Catatan: keluaran bisa berbeda tipis karena model saat training melewati prepare_model_for_kbit_training (beberapa parameter di-upcast), sedangkan model hasil reload tidak.


## 10. Rangkuman dan bahan diskusi

**Yang dikerjakan:** hitungan parameter LoRA per arsitektur, pengukuran memori 4-bit, dua studi kasus QLoRA, evaluasi dengan tiga set prompt, perbandingan epoch, serta simpan-muat-pasang-lepas adapter.

**Diskusi**
- Dari tabel hasil: aspek mana yang cepat dipelajari (format, nama brand, fakta), dan mana yang tetap rapuh?
- Untuk informasi yang berubah (harga, kebijakan retur, jam operasional), mana yang lebih tepat lewat **RAG** dan mana lewat fine-tuning?
- Apa risikonya jika fakta perusahaan "tertanam" di bobot adapter lalu kebijakan berubah?
- Bagaimana evaluasi yang lebih sistematis: test set lebih besar, beberapa seed, LLM-as-a-judge untuk isi jawaban?

**Eksperimen lanjutan**
- Ubah `r` menjadi 4 dan 64 lalu bandingkan dengan tabel bagian 2.
- Perbanyak fakta dan variasi data, lalu lihat apakah `fact_pct` pada `paraphrase` ikut naik.
- Ganti model ke ukuran 1.5B dan bandingkan kebutuhan VRAM dengan perkiraan di bagian 3.

---

*Materi berdasarkan kurikulum Machine Learning on Production, rubythalib.ai.*